# 沖縄県用Jageocoder辞書 作成ツール（build_okinawa_jageocoder）

`jageocoder-converter` を使って **沖縄県（都道府県コード47）のみ** のJageocoderローカル辞書を作成し、
Google Driveへ保存するための補助Notebookです。

**このNotebookは`sheltermatch.ipynb`本体ではありません。**
`sheltermatch.ipynb` は住所→座標変換（ジオコーディング）にJageocoderのローカル辞書を利用しますが、
その辞書を作る処理はここに分離しています。作成した辞書のパスを `sheltermatch.ipynb` の
`JAGEOCODER_DB_DIR` に設定して使ってください。

**重要: このNotebookは毎回実行するものではありません。**
辞書を新規作成する時、または更新したい時だけ実行してください。全国版に比べれば小規模ですが、
辞書生成には数分〜数十分かかることがあります。既に辞書がある場合は、既定では上書きしません
（詳細は「4. 出力先設定」を参照）。

## 2. 必要ライブラリのインストール

Google Colabに標準で入っていない `jageocoder` と `jageocoder-converter` をインストールします。

In [ ]:
%pip install -q jageocoder jageocoder-converter

## 3. Google Driveのマウント

作成した辞書を保存するため、Google Driveをマウントします。表示される認証手順に従ってください。

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
print("Google Driveをマウントしました。")

## 4. 出力先設定

このNotebookで編集が必要な設定はこのセルだけです。

In [ ]:
# ===== 設定（このセルの値を必要に応じて編集してください） =====

# 生成する辞書の対象都道府県コード（47 = 沖縄県）
PREF_CODE = "47"

# 辞書の出力先ディレクトリ（Google Drive上のパス）。
# sheltermatch.ipynb の JAGEOCODER_DB_DIR には、ここで指定したパスをそのまま設定してください。
JAGEOCODER_DB_DIR = "/content/drive/MyDrive/jageocoder/okinawa_db"

# 出力先に既に辞書がある場合に、上書きして再作成するかどうか。
# False（既定）の場合、既存の辞書があれば辞書生成をスキップし、上書きしません。
# 再作成したい場合のみ True に変更してください。
FORCE_REBUILD = False

print("設定を読み込みました。")
print(f"  PREF_CODE         = '{PREF_CODE}'")
print(f"  JAGEOCODER_DB_DIR = '{JAGEOCODER_DB_DIR}'")
print(f"  FORCE_REBUILD     = {FORCE_REBUILD}")

## 5. 沖縄県辞書の生成

`jageocoder-converter` を使い、沖縄県（都道府県コード47）のみを対象に辞書を生成します。
街区・地番レベル以上の精度を確保するため `--no-gaiku` は指定しません。また、通常の住所照合精度を
優先し、住居表示住所データも省略せず、都道府県コード（`47`）のみを指定するシンプルな構成にしています。

元データの利用規約への同意を求められることがあります。その場合は表示内容を確認し、
ご自身で回答してください（このNotebookは同意確認を `--quiet` 等で自動的に省略しません）。

既に出力先に辞書がある場合、`FORCE_REBUILD = False`（既定）のままなら生成をスキップし、
既存の辞書を上書きしません。

In [ ]:
import os

os.makedirs(JAGEOCODER_DB_DIR, exist_ok=True)
existing_files = os.listdir(JAGEOCODER_DB_DIR)

if existing_files and not FORCE_REBUILD:
    print("既存の辞書が見つかりました。意図しない上書きを防ぐため、辞書生成をスキップします。")
    print(f"  辞書ディレクトリ: {JAGEOCODER_DB_DIR}")
    print(f"  既存ファイル/フォルダ数: {len(existing_files)}件")
    print("再作成する場合は、上の「4. 出力先設定」で FORCE_REBUILD = True に変更してから、このセルを再実行してください。")
else:
    if existing_files and FORCE_REBUILD:
        print("FORCE_REBUILD=True のため、既存の辞書を再作成します。")

    print(f"沖縄県（都道府県コード {PREF_CODE}）のJageocoder辞書生成を開始します。")
    print("データ量によっては数分〜数十分かかることがあります。")
    print("元データの利用規約への同意を求められる場合は、表示内容を確認して回答してください。")

    !python -m jageocoder_converter convert --db-dir="{JAGEOCODER_DB_DIR}" {PREF_CODE}

## 6. 生成結果の確認

辞書ファイルが出力先に作成されたか、またJageocoderから読み込めるかを確認します。

In [ ]:
import jageocoder

generated_files = os.listdir(JAGEOCODER_DB_DIR)

if not generated_files:
    print("辞書ディレクトリが空です。生成に失敗している可能性があります。上のセルの出力を確認してください。")
else:
    print(f"辞書ディレクトリにファイルが生成されています（{len(generated_files)}件）。")
    try:
        jageocoder.init(db_dir=JAGEOCODER_DB_DIR)
        print("Jageocoderで辞書を正常に読み込めることを確認しました。")
    except Exception as error:
        print("辞書ファイルは生成されていますが、Jageocoderからの読み込みに失敗しました。")
        print(f"詳細: {error}")

## 7. sheltermatch.ipynb に設定するパスの表示

`sheltermatch.ipynb` の設定セルに貼り付けるべき値を表示します。

In [ ]:
generated_files = os.listdir(JAGEOCODER_DB_DIR)

if generated_files:
    print("沖縄県用Jageocoder辞書を作成しました。\n")
    print("sheltermatch.ipynb の設定:")
    print(f'JAGEOCODER_DB_DIR = "{JAGEOCODER_DB_DIR}"')
else:
    print("辞書ファイルが見つからないため、sheltermatch.ipynbへ設定するパスは表示できません。")
    print("「5. 沖縄県辞書の生成」のセルの出力を確認し、生成が完了しているかを確認してください。")